# Flow-Aware Temporal Pattern Mining for Multi-Stage NIDS — CIC-IDS2017End-to-end walkthrough of Stages 1–9 of the methodology, with every figure andtable displayed inline and written to `results/`.Runs in **smoke mode** by default so the whole notebook executes in minutes. Set`SMOKE = False` for a full run.> Everything learned from data — imputer, scaler, encoder, correlation filter,> the eight rankers, N\*, the calibrators, the fusion weights and τ_R — is fitted> on the training partition, or on validation where the methodology says so.> **The test partition is never used for fitting, selection or tuning.**

## 0. Setup

In [ ]:
import os, sys, warnings, jsonfrom pathlib import Pathwarnings.filterwarnings("ignore")os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")ROOT = Path.cwd()if str(ROOT) not in sys.path:    sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlibmatplotlib.use("module://matplotlib_inline.backend_inline")import matplotlib.pyplot as pltimport utilsimport data_preprocessing as dpimport feature_selection as fselimport imbalance_handling as ihimport models as Mimport fusion as fusimport temporal_pattern_mining as tpmimport evaluation as evimport pipelinepd.set_option("display.width", 160)pd.set_option("display.max_columns", 50)# ---- configure the run -------------------------------------------------DATA_DIR = os.environ.get("NIDS_DATA_DIR", "data/synthetic")SMOKE = TrueOUT = Path("results/notebook")OUT.mkdir(parents=True, exist_ok=True)if not Path(DATA_DIR).exists():    print(f"{DATA_DIR} not found - generating the synthetic CIC-IDS2017 fixture")    from tests.synthetic_data import generate    generate(DATA_DIR, total_rows=40000, seed=7)cfg = utils.load_config("config.yaml", {"run": {"smoke_test": SMOKE,                                                "output_dir": str(OUT)}})SEED = utils.seeded(cfg)utils.set_global_seed(SEED)env = utils.configure_frameworks(cfg)CLASSES = list(range(len(cfg["labels"]["class_order"])))CLASS_NAMES = {i: n for i, n in enumerate(cfg["labels"]["class_order"])}print(f"data_dir  : {DATA_DIR}")print(f"smoke     : {cfg['run']['smoke_test']}")print(f"seed      : {SEED}")print(f"imbalance : {cfg['imbalance']['mode']}   split: {cfg['split']['strategy']}")print(f"fusion    : {cfg['fusion']['mode']}")print(f"env       : torch={env.get('torch')} tf={env.get('tensorflow')} gpu={env.get('gpu')}")

## 1. Config + data load — Stage 1The daily CSVs are concatenated, column names stripped, the 12-hour afternoontimestamps corrected from the file name, labels grouped per the Stage 1.3 table,and the symmetric session key built (eq. 22, see README D5/D6).

In [ ]:
df, feature_columns, prep_report = dp.prepare_dataset(    cfg, data_dir=DATA_DIR, seed=SEED, out_dir=str(OUT), use_cache=True)print(f"rows            : {len(df):,}")print(f"candidate feats : {len(feature_columns)}")print(f"sessions        : {df[dp.COL_SESSION].nunique():,}")print(f"time range      : {df[dp.COL_TIME].min()}  ->  {df[dp.COL_TIME].max()}")print(f"flows/session   : {len(df) / df[dp.COL_SESSION].nunique():.2f}")dist = dp.class_distribution_table(df, cfg)utils.save_table(dist, OUT / "class_distribution.csv")dist

In [ ]:
ev.plot_class_distribution(dist, cfg, OUT / "class_distribution.png")from IPython.display import Image, displaydisplay(Image(str(OUT / "class_distribution.png")))

## 2. Preprocessing report and the split — Stages 2–3`ColumnCleaner` does only what learns nothing from the data (dedup, inf→NaN,negative-sentinel policy) so it may run before the split.`TrainFittedPreprocessor` (median imputer + Min-Max + one-hot) is fitted on the**training rows only** — it refuses anything else.

In [ ]:
print("--- preprocessing report (Stage 2) ---")for k, v in prep_report.items():    if k in ("classes",):        continue    if isinstance(v, list) and len(v) > 8:        v = v[:8] + ["..."]    print(f"  {k:32s} {v}")

In [ ]:
split = dp.make_split(df, cfg, SEED)split_table = dp.split_distribution_table(df, split, cfg)utils.save_table(split_table, OUT / "split_distribution.csv")print(f"train/val/test  : {split.sizes()}")print(f"gap discarded   : {len(split.dropped_gap)} flows "      f"({100*len(split.dropped_gap)/len(df):.2f}%)")print(f"exempt sessions : {len(split.meta.get('exempt_sessions', []))} "      f"(tiny classes blocked at flow level - README D7)")split_table

In [ ]:
# Leakage guards run as assertions, not as comments. Any violation raises.report = utils.assert_no_leakage(    split,    session_ids=df[dp.COL_SESSION].to_numpy(),    timestamps=df[dp.COL_TIME].to_numpy(),    labels=df[dp.COL_LABEL].to_numpy(),)print("leakage checks passed:")for k, v in report.items():    print(f"  [OK] {k}")# The time-blocked ordering, restated explicitly per class.ts = df[dp.COL_TIME].to_numpy()print("\nper-class block boundaries (train_max <= val_min <= test_min):")groups = split.meta["assign_group"]for g in sorted({x for x in np.unique(groups) if x}):    rows = set(np.where(groups == g)[0].tolist())    tr = [i for i in split.train if i in rows]    va = [i for i in split.val if i in rows]    te = [i for i in split.test if i in rows]    if tr and te:        print(f"  {str(g):14s} train_max={pd.Timestamp(ts[tr].max())}  "              f"test_min={pd.Timestamp(ts[te].min())}  "              f"ok={ts[tr].max() <= ts[te].min()}")

In [ ]:
# Fit the transformer on TRAIN only, then transform all three partitions.pre = dp.TrainFittedPreprocessor(cfg).fit(df, feature_columns,                                          rows=split.train, split=split)X_tr = pre.transform(df, split.train)X_va = pre.transform(df, split.val)X_te = pre.transform(df, split.test)y = df[dp.COL_Y].to_numpy()y_tr, y_va, y_te = y[split.train], y[split.val], y[split.test]print(f"X_train {X_tr.shape}   X_val {X_va.shape}   X_test {X_te.shape}")print(f"constant features mapped to a constant: {int(pre.constant_mask_.sum())}")print(f"one-hot columns added: {len(pre.output_columns_) - len(pre.numeric_columns_)}")X_tr.describe().T[["min", "max", "mean", "std"]].head(8)

## 3. Feature selection — Stage 4Redundancy filter → eight rankers (FeS Set-1 … Set-8) → frequency vote ψ ≥ η →normalized-rank aggregation → N\* chosen on **validation** macro-F1.

In [ ]:
fs_result = fsel.run_feature_selection(    X_tr, y_tr, X_va, y_va, cfg, SEED,    session_ids_train=df[dp.COL_SESSION].to_numpy()[split.train],    times_train=df[dp.COL_TIME].to_numpy()[split.train],    out_dir=str(OUT), cache_tag="notebook",)print(f"F_corr after redundancy removal : {len(fs_result.f_corr)}")print(f"pool P (psi >= {cfg['feature_selection']['vote_threshold']})            "      f": {len(fs_result.pool)}")print(f"N* chosen on validation         : {fs_result.n_star}")print(f"\nF* = {fs_result.f_star}")

In [ ]:
# The eight top-K sets (FeS Set-1 ... Set-8)K = cfg["feature_selection"]["top_k"]sets = {fsel.FES_SET_LABELS[m]: fsel.topk_set(s, fs_result.f_corr, K)        for m, s in fs_result.ranker_scores.items()}pd.DataFrame({k: pd.Series(v[:12]) for k, v in sets.items()}).fillna("")

In [ ]:
fs_result.votes_table.head(20)[    ["feature", "psi_votes", "score", "in_pool"]    + [c for c in fs_result.votes_table.columns if c.startswith("in_")]]

In [ ]:
for fn, path in [    (lambda: ev.plot_feature_votes(fs_result.votes_table, cfg, OUT / "feature_votes_heatmap.png"),     OUT / "feature_votes_heatmap.png"),    (lambda: ev.plot_rank_scores(fs_result.votes_table, cfg, OUT / "feature_rank_scores.png"),     OUT / "feature_rank_scores.png"),    (lambda: ev.plot_n_star_curve(fs_result.n_star_curve, fs_result.n_star, cfg,                                  OUT / "n_star_curve.png"), OUT / "n_star_curve.png"),]:    fn()    display(Image(str(path)))

In [ ]:
print("N* search on the VALIDATION partition:")display(fs_result.n_star_curve)if fs_result.stability is not None:    ev.plot_stability(fs_result.stability, cfg, OUT / "feature_stability.png")    display(Image(str(OUT / "feature_stability.png")))    display(fs_result.stability.head(12))

In [ ]:
# Freeze F* and apply it unchanged to all three partitions.F_tr = fs_result.apply(X_tr, "train")F_va = fs_result.apply(X_va, "validation")F_te = fs_result.apply(X_te, "test")utils.assert_feature_mask_frozen(fs_result.f_star, list(F_te.columns), "test")print(f"F* applied unchanged: {F_tr.shape}, {F_va.shape}, {F_te.shape}")

## 4. Imbalance handling — Stage 5Training partition only, **after** feature selection, so synthetic rows cannotinfluence which features were chosen. Validation and test are never resampled.

In [ ]:
n_val_before, n_test_before = len(F_va), len(F_te)X_bal, y_bal, class_weights, resample_report = ih.apply_imbalance_handling(    F_tr, y_tr, cfg, SEED, classes=np.array(CLASSES))# Assert the evaluation partitions came through untouched.ih.assert_eval_sets_untouched(n_val_before, len(F_va), n_test_before, len(F_te))print("[OK] validation and test were not resampled\n")table = resample_report.to_frame(CLASS_NAMES)utils.save_table(table, OUT / "imbalance_report.csv")display(table)print("\npolicy notes:")for n in resample_report.notes:    print("  -", n)if class_weights:    print("\nclass weights w_c = N / (C * N_c):")    print({CLASS_NAMES[k]: round(v, 3) for k, v in class_weights.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))idx = np.arange(len(table))ax.bar(idx - 0.2, table["before"], 0.4, label="before")ax.bar(idx + 0.2, table["after"], 0.4, label="after")ax.set_xticks(idx, table["class"], rotation=30, ha="right")ax.set_yscale("log"); ax.set_ylabel("training rows (log)")ax.set_title(f"Stage 5: class counts before/after ({cfg['imbalance']['mode']})")ax.legend(); ax.grid(axis="y", alpha=0.3)utils.save_fig(fig, OUT / "imbalance_before_after.png", cfg["run"]["figure_dpi"], close=False)plt.show()

## 5. Models — Stage 6RF, XGBoost, LSTM, 1-D CNN and the temporal GNN. The LSTM and GNN train on theun-resampled ordered flows with class weighting (README D10).

In [ ]:
ctx_tr = pipeline._context(df, split.train)ctx_va = pipeline._context(df, split.val)ctx_te = pipeline._context(df, split.test)seq_models = set(cfg["imbalance"].get("sequence_models", ["lstm", "gnn"]))resampled = len(X_bal) != len(F_tr)split_training = resampled and not cfg["imbalance"].get("apply_to_sequence_models", False)flow_cfg = utils.deep_merge(cfg, {"models": {"enabled": [    m for m in cfg["models"]["enabled"] if not (split_training and m in seq_models)]}})trained = M.train_all_models(X_bal, y_bal, F_va, y_va, flow_cfg, CLASSES, SEED,                             class_weights=class_weights,                             train_context=ctx_tr, val_context=ctx_va)if split_training:    seq_cfg = utils.deep_merge(cfg, {"models": {"enabled": [        m for m in cfg["models"]["enabled"] if m in seq_models]}})    w_seq = ih.compute_class_weights(y_tr, cfg, np.array(CLASSES))    trained.update(M.train_all_models(F_tr, y_tr, F_va, y_va, seq_cfg, CLASSES, SEED,                                      class_weights=w_seq,                                      train_context=ctx_tr, val_context=ctx_va))print("trained:", sorted(trained))

In [ ]:
# Training curves for the Keras models and the GNN.curves = {n: m.history_ for n, m in trained.items() if hasattr(m, "history_")}if curves:    fig, axes = plt.subplots(1, len(curves), figsize=(4.6 * len(curves), 3.6), squeeze=False)    for ax, (name, h) in zip(axes[0], curves.items()):        ax.plot(h["loss"], label="train loss")        if "val_loss" in h:            ax.plot(h["val_loss"], label="val loss")        ax.set_title(f"{name} training"); ax.set_xlabel("epoch")        ax.legend(fontsize=8); ax.grid(alpha=0.3)    fig.tight_layout()    utils.save_fig(fig, OUT / "training_curves.png", cfg["run"]["figure_dpi"], close=False)    plt.show()

In [ ]:
val_proba = M.predict_all(trained, F_va, ctx_va)test_proba = M.predict_all(trained, F_te, ctx_te)for name, p in test_proba.items():    utils.assert_probabilities_valid(p, name)print("[OK] every model's probabilities form a simplex over the canonical class order\n")stats = M.model_stats_table(trained)utils.save_table(stats, OUT / "model_efficiency.csv")display(stats)per_model = ev.metrics_to_frame([    ev.evaluate_predictions(y_te, p, CLASSES, cfg, CLASS_NAMES, label=n)    for n, p in test_proba.items()])per_model[["label", "accuracy", "macro_f1", "weighted_f1",           "roc_auc_ovr_macro", "ece"]].sort_values("macro_f1", ascending=False)

## 6. Calibration — Stage 7.1Isotonic/Platt for the tree and LSTM models, temperature scaling for the CNN andGNN. **Fitted on the validation partition only.**

In [ ]:
y_va_idx = np.array([CLASSES.index(v) for v in y_va])calibrators = fus.fit_calibrators(val_proba, y_va_idx, cfg)val_cal = fus.apply_calibrators(val_proba, calibrators)test_cal = fus.apply_calibrators(test_proba, calibrators)rows = []for name in val_proba:    rows.append({        "model": name,        "calibrator": type(calibrators.get(name)).__name__ if name in calibrators else "none",        "ECE before": ev.expected_calibration_error(val_proba[name], y_va_idx),        "ECE after": ev.expected_calibration_error(val_cal[name], y_va_idx),        "Brier before": ev.brier_score(val_proba[name], y_va_idx),        "Brier after": ev.brier_score(val_cal[name], y_va_idx),    })pd.DataFrame(rows).round(4)

In [ ]:
for name in list(val_proba)[:3]:    ev.plot_reliability_diagram(val_proba[name], val_cal.get(name), y_va_idx, cfg,                                OUT / f"reliability_{name}.png",                                title=f"Reliability diagram - {name} (validation)")    display(Image(str(OUT / f"reliability_{name}.png")))

## 7. Adaptive evidence fusion — Stage 7`q_i(t) = λ1·Q_i + λ2·C_i(t) + λ3·K_i(t)`, `w_i(t) = q_i(t) / Σ q_j(t)`.First the Stage 9 evidence that the fusion consumes.

In [ ]:
temporal = tpm.run_temporal_mining(df, split.train, cfg, out_dir=str(OUT),                                   class_names=CLASS_NAMES)use_temporal = cfg["fusion"]["mode"] == "with_temporal"ev_va = temporal.evidence_for(df, split.val, cfg) if use_temporal else Noneev_te = temporal.evidence_for(df, split.test, cfg) if use_temporal else Noneevents_va = temporal.discretiser.transform(df.iloc[split.val]) if temporal.discretiser else Noneevents_te = temporal.discretiser.transform(df.iloc[split.test]) if temporal.discretiser else Noneprev_va = fus.previous_states(df, split.val, cfg, events_va)prev_te = fus.previous_states(df, split.test, cfg, events_te)fusion = fus.EvidenceFusion(cfg=cfg, classes=CLASSES, class_names=CLASS_NAMES,                            calibrators=calibrators, use_temporal=use_temporal,                            graph=temporal.graph)fusion.fit(val_cal, y_va, prev_state=prev_va,           sp=ev_va["sp"] if ev_va else None,           tc=ev_va["tc"] if ev_va else None,           availability=ev_va["availability"] if ev_va else None,           graph=temporal.graph)print(f"baseline reliabilities Q : "      f"{ {k: round(v, 4) for k, v in fusion.Q.items()} }")print(f"lambdas (tuned on val)   : {tuple(round(l, 2) for l in fusion.lambdas)}")print(f"tau_R (tuned on val)     : {fusion.tau_R:.4f}")

In [ ]:
fw = fusion.fuse(test_cal, prev_state=prev_te,                 sp=ev_te["sp"] if ev_te else None,                 tc=ev_te["tc"] if ev_te else None,                 availability=ev_te["availability"] if ev_te else None)utils.assert_weights_sum_to_one(fw.weights, "fusion weights")utils.assert_probabilities_valid(fw.fused_proba, "fused probabilities")print("[OK] weights sum to 1 per session; fused probabilities sum to 1 per row\n")fus.save_fusion_artifacts(fw, fusion, OUT, "notebook")pd.DataFrame({"source": fw.sources,              "mean weight": fw.weights.mean(axis=0).round(4),              "std weight": fw.weights.std(axis=0).round(4),              "min": fw.weights.min(axis=0).round(4),              "max": fw.weights.max(axis=0).round(4)})

In [ ]:
ev.plot_fusion_weights(fw, cfg, OUT / "fusion_weights.png")display(Image(str(OUT / "fusion_weights.png")))

In [ ]:
# A single-session worked example, in the layout of methodology section 7.4.# (The document's own numbers there are illustrative; these are real.)row = int(np.argmax(fw.risk))print(f"worked example - test row {row}, "      f"true class = {CLASS_NAMES[int(y_te[row])]}, "      f"R_t = {fw.risk[row]:.4f}, tau_R = {fusion.tau_R:.4f} "      f"-> {'ALERT' if fw.risk[row] >= fusion.tau_R else 'no alert'}")detail = []for j, src in enumerate(fw.sources):    entry = {"source": src,             "baseline Q": round(fusion.Q.get(src, float('nan')), 4),             "reliability q": round(float(fw.reliabilities[row, j]), 4),             "weight w": round(float(fw.weights[row, j]), 4)}    if src in test_cal:        entry["confidence C"] = round(float(fus.confidence_term(test_cal[src])[row]), 4)        entry["attack score 1-P(Benign)"] = round(float(1 - test_cal[src][row, 0]), 4)    elif src == "sp":        entry["attack score 1-P(Benign)"] = round(float(ev_te["sp"][row]), 4)    elif src == "tc":        entry["attack score 1-P(Benign)"] = round(float(ev_te["tc"][row]), 4)    detail.append(entry)display(pd.DataFrame(detail))print(f"\nfused class = {CLASS_NAMES[int(np.argmax(fw.fused_proba[row]))]}")

In [ ]:
ablations = fus.run_fusion_ablations(    fusion, test_cal, y_te, prev_te,    ev_te["sp"] if ev_te else None, ev_te["tc"] if ev_te else None,    ev_te["availability"] if ev_te else None, cfg)utils.save_table(ablations, OUT / "fusion_ablations.csv")print("Stage 7.6 ablations (baselines, not alternative methods):")ablations

## 8. Evaluation — Stage 8Macro-F1 and per-class recall are the primary metrics; accuracy is dominated bythe benign class.

In [ ]:
proba_for_eval = dict(test_proba)proba_for_eval["fusion"] = fw.fused_probafor n, p in test_cal.items():    proba_for_eval[f"{n}_calibrated"] = preport = ev.full_evaluation(y_te, proba_for_eval, CLASSES, cfg, OUT,                            CLASS_NAMES, tag="notebook", make_plots=True,                            extra_stats=stats)report["metrics"][["label", "accuracy", "macro_f1", "weighted_f1",                   "recall_macro", "roc_auc_ovr_macro", "pr_auc_macro",                   "ece", "brier", "rare_class_recall_macro"]].round(4)

In [ ]:
print("per-class results for the fusion (rare classes flagged):")pc = report["per_class"]pc[pc["label"] == "fusion"][["class", "support", "precision", "recall", "f1", "is_rare"]].round(4)

In [ ]:
best = report["metrics"].iloc[0]["label"]for p in [OUT / f"confusion_fusion_notebook.png",          OUT / f"confusion_fusion_norm_notebook.png",          OUT / "roc_curves_notebook.png",          OUT / "pr_curves_notebook.png"]:    if p.exists():        display(Image(str(p)))

In [ ]:
if len(report["significance"]):    print("pairwise significance tests "          f"({cfg['evaluation']['significance']['test']}):")    display(report["significance"].head(15).round(5))

In [ ]:
# SHAP-LGBM ranker summary (FeS Set-4)if "shap_lgbm" in fs_result.ranker_scores:    ev.plot_shap_summary(fs_result.ranker_scores["shap_lgbm"], fs_result.f_corr,                         cfg, OUT / "shap_summary.png")    display(Image(str(OUT / "shap_summary.png")))    top = np.argsort(-fs_result.ranker_scores["shap_lgbm"])[:12]    display(pd.DataFrame({"feature": [fs_result.f_corr[i] for i in top],                          "mean |SHAP|": fs_result.ranker_scores["shap_lgbm"][top]}).round(5))

## 9. Temporal pattern mining — Stage 9Mined on TRAIN sequences only. See README D15–D19 for the choices this stagemakes, since the methodology names it but does not specify it.

In [ ]:
print(f"FP-Growth itemsets : {len(temporal.fp_itemsets)}")print(f"PrefixSpan patterns: {len(temporal.sequences_patterns)}")print(f"attack-state graph : {len(temporal.graph.states)} states, "      f"{int(temporal.graph.counts.sum())} transitions observed")display(temporal.sequences_patterns.head(12)[["pattern", "length", "count", "support"]])

In [ ]:
for fn, path in [    (lambda: ev.plot_top_patterns(temporal.sequences_patterns, cfg, OUT / "top_patterns.png"),     OUT / "top_patterns.png"),    (lambda: ev.plot_attack_state_graph(temporal.graph, cfg, OUT / "attack_state_graph.png"),     OUT / "attack_state_graph.png"),]:    fn()    if path.exists():        display(Image(str(path)))

In [ ]:
# Transition matrix of G_Ttm = pd.DataFrame(temporal.graph.matrix, index=temporal.graph.states,                  columns=temporal.graph.states)display(tm.round(3).iloc[:10, :10])if ev_te is not None:    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))    axes[0].hist(ev_te["sp"], bins=30, color="#7a5195"); axes[0].set_title("$SP_t$ (test)")    axes[1].hist(ev_te["tc"], bins=30, color="#2a7f3f"); axes[1].set_title("$TC_t$ (test)")    axes[2].hist(fw.risk, bins=30, color="#d1495b")    axes[2].axvline(fusion.tau_R, color="k", ls="--", label=f"$\\tau_R$={fusion.tau_R:.2f}")    axes[2].set_title("fused risk $R_t$ (test)"); axes[2].legend()    for a in axes:        a.grid(alpha=0.3)    fig.tight_layout()    utils.save_fig(fig, OUT / "temporal_distributions.png", cfg["run"]["figure_dpi"], close=False)    plt.show()    print(f"sessions with temporal evidence available: "          f"{ev_te['availability'].mean():.1%}")

## 10. Branch comparison and the leakage gapBranch D's point: a plain flow-level stratified split tears sessions apart andputs near-duplicate flows on both sides. The drop from that number to thetime-blocked number is how much of a reported score comes from the evaluationprotocol rather than from the detector.

In [ ]:
gap_rows = []df_cache = (df, feature_columns, prep_report)for strategy in ["stratified_flow", "stratified_session", "time_blocked"]:    c = utils.deep_merge(cfg, {"split": {"strategy": strategy},                               "run": {"name": f"nb_{strategy}",                                       "output_dir": str(OUT / strategy)}})    r = pipeline.run_pipeline(DATA_DIR, cfg=c, out_dir=c["run"]["output_dir"],                              tag=strategy, make_plots=False, df_cache=df_cache)    fus_row = r.metrics.loc[r.metrics["label"] == "fusion"]    gap_rows.append({        "split_strategy": strategy,        "fusion_macro_f1": float(fus_row["macro_f1"].iloc[0]) if len(fus_row) else None,        "fusion_accuracy": float(fus_row["accuracy"].iloc[0]) if len(fus_row) else None,        "best_macro_f1": float(r.metrics.iloc[0]["macro_f1"]),        "n_star": r.feature_result.n_star,        **{f"n_{k}": v for k, v in r.split.sizes().items()},    })gap = pd.DataFrame(gap_rows)ref = gap.loc[gap["split_strategy"] == "time_blocked", "fusion_macro_f1"].iloc[0]gap["leakage_gap_vs_time_blocked"] = gap["fusion_macro_f1"] - refutils.save_table(gap, OUT / "leakage_gap.csv")gap.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))colors = ["#d1495b", "#edae49", "#2a7f3f"]ax.bar(gap["split_strategy"], gap["fusion_macro_f1"], color=colors)for i, v in enumerate(gap["fusion_macro_f1"]):    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)ax.set_ylabel("fusion macro-F1 (test)")ax.set_title("Leakage gap: the same pipeline under three evaluation protocols")ax.grid(axis="y", alpha=0.3)utils.save_fig(fig, OUT / "leakage_gap.png", cfg["run"]["figure_dpi"], close=False)plt.show()print(f"The flow-level protocol reports "      f"{gap.iloc[0]['leakage_gap_vs_time_blocked']:+.4f} macro-F1 relative to "      f"the time-blocked protocol on identical data.")

In [ ]:
# If run_all.py has been executed, show the aggregated branch comparison.comparison = Path("results/branch_comparison.csv")if comparison.exists():    display(pd.read_csv(comparison))else:    print("Run `python run_all.py --data_dir <path> --smoke_test` to produce "          "results/branch_comparison.csv across all five branches.")

## Stage checklist

In [ ]:
checks = [    ("Stage 1: dataset consolidation", "data_preprocessing.py", OUT / "class_distribution.csv"),    ("Stage 2: preprocessing",         "data_preprocessing.py", OUT / "preprocessing_report.json"),    ("Stage 3: splits",                "data_preprocessing.py", OUT / "split_distribution.csv"),    ("Stage 4: feature selection",     "feature_selection.py",  OUT / "feature_votes.csv"),    ("Stage 5: imbalance handling",    "imbalance_handling.py", OUT / "imbalance_report.csv"),    ("Stage 6: models",                "models.py",             OUT / "model_efficiency.csv"),    ("Stage 7: evidence fusion",       "fusion.py",             OUT / "fusion_weights_notebook.csv"),    ("Stage 8: evaluation",            "evaluation.py",         OUT / "metrics_notebook.csv"),    ("Stage 9: temporal mining",       "temporal_pattern_mining.py", OUT / "attack_state_graph.json"),    ("Leakage gap (branch D)",         "branches/branch_D_stratified_split/run.py", OUT / "leakage_gap.csv"),]print(utils.format_checklist([(s, f, "OK" if Path(a).exists() else "MISSING")                              for s, f, a in checks]))